In [2]:
from mini_competition.data_functions import load_data, add_zero_sales, PROCCESED_PATH
import pandas as pd

data = load_data(PROCCESED_PATH / "train.parquet" )
data = data[data["date"] > "2016-07-01"]

In [3]:
# add zeros
data = add_zero_sales(data)

In [ ]:
import numpy as np

# add lag features
def add_lag_features(df : pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(["store_nbr", "item_nbr", "date"]).reset_index(drop=True)
    df["log_sales"] = np.log1p(df["unit_sales"].astype("float32"))

    g = df.groupby(["store_nbr", "item_nbr"])["log_sales"]

    for lag in [16, 21, 28, 35]:
        df[f"lag_{lag}"] = g.shift(lag)

    df["rolling_mean_7"]  = g.transform(lambda s: s.shift(16).rolling(7).mean())
    df["rolling_mean_28"] = g.transform(lambda s: s.shift(16).rolling(28).mean())

    return df.copy()

def drop_lag_none_rows(df : pd.DataFrame) -> pd.DataFrame:
    cols = ["lag_16", "lag_21","lag_28", "lag_35","rolling_mean_7", "rolling_mean_28"]
    df = df.dropna(subset=cols, how="any")

data_with_lags = drop_lag_none_rows(add_lag_features(data))

NameError: name 'np' is not defined

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import TargetEncoder, FunctionTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
import numpy as np

def clean_features(d):
    d = d.copy()
    d["onpromotion"] = d["onpromotion"].fillna(False)
    return d

def add_date_features(d):
    d = d.copy()
    # adds weekday and month feature
    d["date"] = pd.to_datetime(d["date"])
    d["weekday_num"] = d["date"].dt.day_of_week
    d["month_num"] = d["date"].dt.month
    return d.drop(columns="date")


df = data_with_lags

train = df[(df["date"] < "2017-07-31") & (df["date"] > "2016-07-31")].sample(n=2_000_000, random_state=0)
valid = df[df["date"] >= "2017-07-31"].sample(n=400_000, random_state=0)

# Use the raw item_nbr and the raw unit_sales; the pipeline handles both
features = train.columns.drop(["date", "unit_sales"])
X_train, y_train = train[features], train["unit_sales"]
X_valid, y_valid = valid[features], valid["unit_sales"]

# Target-encode item_nbr and pass the other columns through unchanged
preprocess = ColumnTransformer(
    [("item_te", TargetEncoder(), ["item_nbr"])],
    remainder="passthrough",
    verbose_feature_names_out=False,
).set_output(transform="pandas")   # keeps store_nbr as 'category'

model = TransformedTargetRegressor(
    regressor=make_pipeline(
        FunctionTransformer(clean_features),
        FunctionTransformer(add_date_features),
        preprocess,
        HistGradientBoostingRegressor(
            categorical_features=["store_nbr"],
            max_iter=1000, learning_rate=0.05, max_leaf_nodes=127,
            early_stopping=True,
        ),
    ),
    func=np.log1p,
    inverse_func=np.expm1,
    check_inverse=False,   # clipping can't be undone, so skip the round-trip check
)


model.fit(X_train, y_train)
pred = model.predict(X_valid)

KeyboardInterrupt: 